# Export QC'd daily rainfall to Station Exchange Format (SEF)

The located, quality-controlled daily rainfall observations are shared with
others in the [Station Exchange Format](https://datarescue.climate.copernicus.eu/station-exchange-format-sef)
(SEF): a simple tab-separated text format where **one file holds one variable
from one station**.

Here each ensemble transcription file is a single *station-year* of daily
rainfall, so the export writes **one SEF `.tsv` per located station-year** under
`<output_root>/tsv/<year>/<ID>.tsv`. Each file carries every day's consensus
daily total (the member median), converted from the original inches to
millimetres, with the QC verdicts travelling in each observation's `Meta`
column (`qc1=...` from the exact-monthly check, `qc2=...` from the secondary
XGBoost check).

This notebook runs a small local export, inspects a generated file, validates
its structure, and shows how the same export runs at scale on SLURM.

## Setup roots and imports

In [2]:
import os
from pathlib import Path

from src.rainfall_rescue_sqlite.sef_export import (
    export_sef,
    default_sef_output_root,
    SEF_VERSION,
    VBL,
    STAT,
    UNITS,
    PERIOD,
    INCHES_TO_MM,
    HEADER_ORDER,
    DATA_COLUMNS,
)
from src.rainfall_rescue_sqlite.parquet_regional_stats import (
    default_daily_consensus_parquet_root,
)
from src.rainfall_rescue_sqlite.parquet_similarity import (
    default_comparison_parquet_root,
)

# Cap DuckDB memory and give it a disk spill dir so the local run cannot OOM the
# workstation (the ordered join spills to disk instead of crashing).
os.environ.setdefault("PDIR", "/data/scratch/philip.brohan/ADRQ")
os.environ.setdefault("DUCKDB_MEMORY_LIMIT", "3GB")
os.environ.setdefault("DUCKDB_TEMP_DIR", "/var/tmp/duckdb_sef_export")
Path(os.environ["DUCKDB_TEMP_DIR"]).mkdir(parents=True, exist_ok=True)

pdir = Path(os.environ["PDIR"])
comparison_root = default_comparison_parquet_root()
consensus_root = default_daily_consensus_parquet_root()

print(f"PDIR:            {pdir}")
print(f"comparison root: {comparison_root}")
print(f"consensus root:  {consensus_root}")
print(f"SEF output root: {default_sef_output_root()}")
print(f"SEF version:     {SEF_VERSION}  (Vbl={VBL}, Stat={STAT}, Units={UNITS}, Period={PERIOD})")

PDIR:            /data/scratch/philip.brohan/ADRQ
comparison root: /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet
consensus root:  /data/scratch/philip.brohan/ADRQ/daily_consensus_parquet
SEF output root: /data/scratch/philip.brohan/ADRQ/sef_export
SEF version:     1.0.0  (Vbl=rr, Stat=sum, Units=mm, Period=1day)


## Run a small local export

Export a slice of the first few thousand `file_id`s into a demo directory under
`/var/tmp`. `export_sef` streams the consensus/metadata/QC join in `file_id`
order and flushes one `.tsv` each time the `file_id` advances, so memory stays
bounded regardless of how many stations fall in the slice.

In [3]:
demo_root = Path("/var/tmp/sef_demo")

result = export_sef(
    output_root=demo_root,
    start_file_id=1,
    end_file_id=2000,
)
result

SEFExportResult(output_root=PosixPath('/var/tmp/sef_demo'), qc_session_id=2, files_written=1160, obs_rows=431520, start_file_id=1, end_file_id=2000)

## Inspect a generated SEF file

Each file has 12 header lines (`name<TAB>value`), then the data-table column
header, then one line per observed day.

In [4]:
tsv_files = sorted(demo_root.glob("tsv/*/*.tsv"))
print(f"{len(tsv_files)} SEF files written to {demo_root / 'tsv'}")

sample = tsv_files[0]
lines = sample.read_text(encoding="utf-8").splitlines()
print(f"\nSample file: {sample.relative_to(demo_root)}  ({len(lines)} lines)\n")
print("\n".join(lines[:21]))
print("...")

1160 SEF files written to /var/tmp/sef_demo/tsv

Sample file: tsv/1866/DRain_1861-1870_Radnorshire-1.tsv  (385 lines)

SEF	1.0.0
ID	DRain_1861-1870_Radnorshire-1
Name	RHAYADER CEFNFAES
Lat	52.3121
Lon	-3.5163
Alt	268.2
Source	RainfallRescue
Link	NA
Vbl	rr
Stat	sum
Units	mm
Meta	orig.units=in|match.type=exact|qc.session=2
Year	Month	Day	Hour	Minute	Period	Value	Meta
1866	1	1	9	0	1day	7.4	qc1=fail|qc2=indeterminate
1866	1	2	9	0	1day	23.1	qc1=fail|qc2=indeterminate
1866	1	3	9	0	1day	2.3	qc1=fail|qc2=indeterminate
1866	1	4	9	0	1day	6.3	qc1=fail|qc2=indeterminate
1866	1	5	9	0	1day	7.4	qc1=fail|qc2=indeterminate
1866	1	6	9	0	1day	4.8	qc1=fail|qc2=indeterminate
1866	1	7	9	0	1day	14.5	qc1=fail|qc2=indeterminate
1866	1	8	9	0	1day	1.5	qc1=fail|qc2=indeterminate
...


## Validate the file structure

Confirm the 12 header lines are in the required order, the data-table header is
correct, every day round-trips back as a row, and the QC verdicts are present.

In [5]:
import pandas as pd

header = dict(line.split("\t", 1) for line in lines[:12])
assert list(header) == HEADER_ORDER, list(header)
assert lines[12].split("\t") == DATA_COLUMNS, lines[12]

df = pd.read_csv(sample, sep="\t", skiprows=12)
assert len(df) == len(lines) - 13, (len(df), len(lines))
assert (df["Value"] >= 0).all()
assert df["Meta"].str.startswith("qc1=").all()

print(f"Station:    {header['ID']}  ({header['Name']})")
print(f"Location:   lat={header['Lat']}, lon={header['Lon']}, alt={header['Alt']} m")
print(f"File Meta:  {header['Meta']}")
print(f"Days:       {len(df)}")
print(df["Value"].describe().round(2).to_string())

Station:    DRain_1861-1870_Radnorshire-1  (RHAYADER CEFNFAES)
Location:   lat=52.3121, lon=-3.5163, alt=268.2 m
File Meta:  orig.units=in|match.type=exact|qc.session=2
Days:       372
count    372.00
mean       3.47
std        5.28
min        0.00
25%        0.00
50%        0.80
75%        5.10
max       32.50


### QC verdict distribution across the demo export

`qc1` is the exact-monthly check verdict for every day; `qc2` is the secondary
XGBoost verdict, which only exists (`pass`/`fail`/`indeterminate`) for the days
that failed `qc1` and were re-examined — it is `NA` otherwise.

In [6]:
from collections import Counter

combos = Counter()
for path in tsv_files:
    for line in path.read_text(encoding="utf-8").splitlines()[13:]:
        combos[line.rsplit("\t", 1)[-1]] += 1

total = sum(combos.values())
print(f"{total} observations across {len(tsv_files)} files\n")
for meta, count in sorted(combos.items()):
    print(f"{count:9d}  ({100 * count / total:5.1f}%)  {meta}")

431520 observations across 1160 files

     7677  (  1.8%)  qc1=fail|qc2=fail
     2449  (  0.6%)  qc1=fail|qc2=indeterminate
   190444  ( 44.1%)  qc1=fail|qc2=pass
   230950  ( 53.5%)  qc1=pass|qc2=NA


## Running at scale on SLURM

The full ~500k-station-year export runs as a single SLURM array (no merge stage
— each shard writes disjoint year-partitioned `.tsv` files):

```bash
scripts/slurm/submit_sef_export.sh
```

Each array task runs `scripts/export_sef.py` for a contiguous `file_id` slice
(`--num-shards` / `--shard-index` / `--total-file-ids`). Shard count and
resources are configured in `scripts/slurm/config.sh` (the `SEF_*` block:
`SEF_NUM_SHARDS`, `SEF_TOTAL_FILE_IDS`, `SEF_SOURCE`, `SEF_LINK`,
`SEF_OBS_HOUR`, `SEF_CORES` / `SEF_MEM_MB` / `SEF_TIME_MIN`).

Prerequisites are the daily-consensus table (`submit_daily_consensus.sh`) and
the located ensemble metadata (`assign_ensemble_metadata`); the QC tables are
joined when present. Files land in
`$PDIR/sef_export/tsv/<year>/<ID>.tsv` with per-shard manifests in
`$PDIR/sef_export/manifests`.

The demo output above lives under `/var/tmp/sef_demo`; remove it with
`rm -rf /var/tmp/sef_demo` when finished.